In [ ]:
# Problema: Diseñar un mart analítico que permita analizar ventas por tiempo, cliente y producto sin confundirlo con la fuente operacional.

"""Crea una fuente operacional y un mart estrella determinista."""
import sqlite3
from pathlib import Path

ROOT = next(path for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (path / "data").is_dir() and (path / "submission").is_dir())
SOURCE, MART = ROOT / "data/sales_operational.db", ROOT / "submission/sales_mart.db"
CUSTOMERS = [("C001","Ana Ruiz","Bogota"),("C002","Luis Gomez","Medellin"),("C003","Mariana Torres","Cali")]
PRODUCTS = [("P100","Cafe","Alimentos",18),("P200","Panaderia","Alimentos",9.5),("P300","Cuaderno","Oficina",7.25),("P400","Lapicero","Oficina",2.5)]
ORDERS = [("O4001","2026-01-05","C001"),("O4002","2026-01-18","C002"),("O4003","2026-02-04","C001"),("O4004","2026-02-20","C003"),("O4005","2026-03-09","C002")]
ITEMS = [("O4001","P100",2),("O4001","P200",1),("O4002","P300",3),("O4003","P400",4),("O4003","P100",1),("O4004","P200",2),("O4004","P300",1),("O4005","P100",3)]


def build_submission():
    for path in (SOURCE, MART): path.unlink(missing_ok=True)
    with sqlite3.connect(SOURCE) as db:
        db.executescript("CREATE TABLE customers(customer_id TEXT PRIMARY KEY, customer_name TEXT, customer_city TEXT); CREATE TABLE products(product_id TEXT PRIMARY KEY, product_name TEXT, category TEXT, unit_price REAL); CREATE TABLE orders(order_id TEXT PRIMARY KEY, order_date TEXT, customer_id TEXT); CREATE TABLE order_items(order_id TEXT, product_id TEXT, quantity INTEGER, PRIMARY KEY(order_id, product_id));")
        db.executemany("INSERT INTO customers VALUES (?, ?, ?)", CUSTOMERS); db.executemany("INSERT INTO products VALUES (?, ?, ?, ?)", PRODUCTS); db.executemany("INSERT INTO orders VALUES (?, ?, ?)", ORDERS); db.executemany("INSERT INTO order_items VALUES (?, ?, ?)", ITEMS)
        lines = db.execute("SELECT o.order_id,o.order_date,c.customer_id,c.customer_name,c.customer_city,p.product_id,p.product_name,p.category,i.quantity,p.unit_price FROM orders o JOIN customers c USING(customer_id) JOIN order_items i USING(order_id) JOIN products p USING(product_id) ORDER BY o.order_id,p.product_id").fetchall()
    with sqlite3.connect(MART) as db:
        db.executescript("CREATE TABLE dim_customer(customer_key INTEGER PRIMARY KEY, customer_id TEXT UNIQUE, customer_name TEXT, customer_city TEXT); CREATE TABLE dim_product(product_key INTEGER PRIMARY KEY, product_id TEXT UNIQUE, product_name TEXT, product_category TEXT); CREATE TABLE dim_date(date_key INTEGER PRIMARY KEY, full_date TEXT UNIQUE, year INTEGER, month INTEGER); CREATE TABLE fact_sales(order_id TEXT, date_key INTEGER REFERENCES dim_date, customer_key INTEGER REFERENCES dim_customer, product_key INTEGER REFERENCES dim_product, quantity INTEGER, unit_price REAL, sales_amount REAL);")
        customers = sorted({row[2:5] for row in lines}); products = sorted({(row[5],row[6],row[7]) for row in lines}); dates = sorted({row[1] for row in lines})
        db.executemany("INSERT INTO dim_customer(customer_id,customer_name,customer_city) VALUES (?, ?, ?)", customers); db.executemany("INSERT INTO dim_product(product_id,product_name,product_category) VALUES (?, ?, ?)", products); db.executemany("INSERT INTO dim_date(full_date,year,month) VALUES (?, ?, ?)", [(d,int(d[:4]),int(d[5:7])) for d in dates])
        for order_id,date,cid,_,_,pid,_,_,qty,price in lines:
            keys = db.execute("SELECT d.date_key,c.customer_key,p.product_key FROM dim_date d,dim_customer c,dim_product p WHERE d.full_date=? AND c.customer_id=? AND p.product_id=?", (date,cid,pid)).fetchone()
            db.execute("INSERT INTO fact_sales VALUES (?, ?, ?, ?, ?, ?, ?)", (order_id,*keys,qty,price,qty*price))


if __name__ == "__main__": build_submission()